In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import igraph
import scvelo as scv
import loompy as lmp
import anndata as ad
import harmonypy as hm
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import ktplotspy as kpy
import holoviews as hv
from holoviews import opts

import celltypist
from celltypist import models
from celltypist.models import Model

import scanpy as sc
from cellphonedb.src.core.methods import cpdb_analysis_method
from cellphonedb.src.core.methods import cpdb_statistical_analysis_method
import os
import pandas as pd
import glob
from IPython.display import HTML, display
from cellphonedb.utils import db_releases_utils
display(HTML(db_releases_utils.get_remote_database_versions_html()['db_releases_html_table']))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from cellphonedb.src.core.methods import cpdb_analysis_method
from cellphonedb.src.core.methods import cpdb_statistical_analysis_method
import os
import pandas as pd
import glob
from IPython.display import HTML, display
from cellphonedb.utils import db_releases_utils
display(HTML(db_releases_utils.get_remote_database_versions_html()['db_releases_html_table']))

# cellphoneDB method 1 & 2

In [ ]:
adata_scrna=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_monoMAC_detail.h5ad")
adata_scrna

In [ ]:
sc.pl.umap(
    adata_scrna,
    color='majority_voting',
    legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    #title='Cell type prediction',
    size=10
)

In [ ]:
adata_scrna.obs['cellannot'] = adata_scrna.obs['majority_voting'].replace({
    'T cells': 'Lymphoid',
    'Resident NK': 'Lymphoid',
    'Circulating NK/NKT': 'Lymphoid',
    'Macrophages': 'Myeloid',
    'Mono+mono derived cells': 'Myeloid',
    'Endothelial cells': 'Endothelial',
    'Fibroblasts': 'Stellete',
    'Hepatocytes': 'Hepatocyte',
    'Cholangiocytes' : 'Cholangiocyte'
})

sc.pl.umap(adata_scrna, color=['cellannot'])

In [ ]:
desired_order = ["Hepatocyte", "Cholangiocyte", "Endothelial", "Stellete", "Lymphoid", "Myeloid","B cells","Basophils","Plasma cells", "pDCs" ]
adata_scrna.obs['cellannot'] = pd.Categorical(adata_scrna.obs['cellannot'], categories=desired_order, ordered=True)

In [ ]:
adata_scrna.obs['detail_cluster'] = adata_scrna.obs['majority_voting'].astype(str)

In [ ]:
adata=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono03.h5ad")

In [ ]:
adata

In [ ]:
adata.obs["cellannot"].value_counts()

In [ ]:
for cell, sub in adata.obs["monoMAC"].items():
    adata_scrna.obs.at[cell, "detail_cluster"] = f"{sub}"

In [ ]:
print(adata_scrna.obs['detail_cluster'].value_counts())

In [ ]:
sc.pl.umap(
    adata_scrna,
    color='detail_cluster',
    legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    #title='Cell type prediction',
    size=10
)

In [ ]:
adata_scrna.write("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_cellphoneDB.h5ad")

In [ ]:
#Define the version and the path to download database
# -- Version of the databse
cpdb_version = 'v5.0.0'

# -- Path where the input files to generate the database are located
cpdb_target_dir = os.path.join('/home/jaehyunchoi/Tool_testing/cellphoneDB/db/test', cpdb_version)

In [ ]:
from cellphonedb.utils import db_utils

db_utils.download_database(cpdb_target_dir, cpdb_version)

In [ ]:
# -- Path where the input files to generate the database are located
cpdb_input_dir = '/home/jaehyunchoi/Tool_testing/cellphoneDB/db/test/v5.0.0/' 
os.listdir(cpdb_input_dir)

In [ ]:
from cellphonedb.utils import db_utils

# -- Creates new database
db_utils.create_db(cpdb_input_dir)

In [ ]:
import pandas as pd
import sys
import os

pd.set_option('display.max_columns', 100)
print(sys.version)

In [ ]:
import pandas as pd

# AnnData 객체에서 celltype 정보 추출
meta = pd.DataFrame({
    "Cell": adata_scrna.obs_names,           # cell barcode
    "cell_type": adata_scrna.obs["detail_cluster"] # cell type/cluster label (여기 col 이름은 바꿔주세요!)
})

# tsv로 저장 (CellPhoneDB에서 요구하는 포맷)
meta.to_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/cell_cellphondeDB_metadata.tsv", sep="\t", index=False)

In [ ]:
cpdb_file_path = '/home/jaehyunchoi/Tool_testing/cellphoneDB/db/test/v5.0.0/cellphonedb.zip'
meta_file_path = '/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/cell_cellphondeDB_metadata.tsv'
counts_file_path = '/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_raw.h5ad'
#microenvs_file_path = 'data/microenvironment.tsv'
out_path = '/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/cellphoneDB/method1'

In [ ]:
metadata = pd.read_csv(meta_file_path, sep = '\t')
metadata.head(3)

In [ ]:
adata = ad.read_h5ad(counts_file_path)
adata.shape

In [ ]:
list(adata.obs.index).sort() == list(metadata['Cell']).sort()

In [ ]:
adata.var_names

In [ ]:
from cellphonedb.src.core.methods import cpdb_analysis_method

cpdb_results = cpdb_analysis_method.call(
    cpdb_file_path = cpdb_file_path,           # mandatory: CellphoneDB database zip file.
    meta_file_path = meta_file_path,           # mandatory: tsv file defining barcodes to cell label.
    counts_file_path = counts_file_path,       # mandatory: normalized count matrix - a path to the counts file, or an in-memory AnnData object
    counts_data = 'hgnc_symbol',               # defines the gene annotation in counts matrix.
    #microenvs_file_path = microenvs_file_path, # optional (default: None): defines cells per microenvironment.
    score_interactions = True,                 # optional: whether to score interactions or not. 
    output_path = out_path,                    # Path to save results    microenvs_file_path = None,
    separator = '|',                           # Sets the string to employ to separate cells in the results dataframes "cellA|CellB".
    threads = 5,                               # number of threads to use in the analysis.
    threshold = 0.1,                           # defines the min % of cells expressing a gene for this to be employed in the analysis.
    result_precision = 3,                      # Sets the rounding for the mean values in significan_means.
    debug = False,                             # Saves all intermediate tables emplyed during the analysis in pkl format.
    output_suffix = None                       # Replaces the timestamp in the output files by a user defined string in the  (default: None)
)

In [ ]:
import pickle
print(cpdb_results.keys())
with open("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/cellphoneDB/method1/cpdb_method1_results.pkl", "wb") as f:
    pickle.dump(cpdb_results, f)

In [ ]:
cpdb_results['means_result'].head(2)

In [ ]:
list(cpdb_results.keys())

In [ ]:
# 결과 불러오기
means = cpdb_results['means_result']

# cluster pair 열만 선택 (annotation 열 제외)
annotation_cols = [
    'id_cp_interaction','interacting_pair','partner_a','partner_b',
    'gene_a','gene_b','secreted','receptor_a','receptor_b',
    'annotation_strategy','is_integrin','directionality','classification'
]
interaction_cols = [c for c in means.columns if c not in annotation_cols]

# mono3가 포함된 interaction 열만 선택
mono_cols = [c for c in interaction_cols if "mono3" in c]

# subset dataframe 만들기
mono_subset = means[annotation_cols + mono_cols]

In [ ]:
mono_subset

In [ ]:
# heatmap에 맞게 index를 interaction 이름으로 설정
heatmap_df = mono_subset.set_index("interacting_pair")[mono_cols]

plt.figure(figsize=(12, 8))
sns.heatmap(
    heatmap_df,
    cmap="Reds",
    linewidths=0,
    linecolor="gray"
)
plt.title("CellPhoneDB mean expression (monoMAC-5 interactions)")
plt.xlabel("Cluster pairs")
plt.ylabel("Interacting pairs")
plt.tight_layout()
plt.show()

In [ ]:
from cellphonedb.src.core.methods import cpdb_statistical_analysis_method
# Method2 

cpdb_results = cpdb_statistical_analysis_method.call(
    cpdb_file_path = cpdb_file_path,                 # mandatory: CellphoneDB database zip file.
    meta_file_path = meta_file_path,                 # mandatory: tsv file defining barcodes to cell label.
    counts_file_path = counts_file_path,             # mandatory: normalized count matrix - a path to the counts file, or an in-memory AnnData object
    counts_data = 'hgnc_symbol',                     # defines the gene annotation in counts matrix.
    #active_tfs_file_path = active_tf_path,           # optional: defines cell types and their active TFs.
    #microenvs_file_path = microenvs_file_path,       # optional (default: None): defines cells per microenvironment.
    score_interactions = True,                       # optional: whether to score interactions or not. 
    iterations = 1000,                               # denotes the number of shufflings performed in the analysis.
    threshold = 0.1,                                 # defines the min % of cells expressing a gene for this to be employed in the analysis.
    threads = 5,                                     # number of threads to use in the analysis.
    debug_seed = 42,                                 # debug randome seed. To disable >=0.
    result_precision = 3,                            # Sets the rounding for the mean values in significan_means.
    pvalue = 0.05,                                   # P-value threshold to employ for significance.
    subsampling = False,                             # To enable subsampling the data (geometri sketching).
    subsampling_log = False,                         # (mandatory) enable subsampling log1p for non log-transformed data inputs.
    subsampling_num_pc = 100,                        # Number of componets to subsample via geometric skectching (dafault: 100).
    subsampling_num_cells = 1000,                    # Number of cells to subsample (integer) (default: 1/3 of the dataset).
    separator = '|',                                 # Sets the string to employ to separate cells in the results dataframes "cellA|CellB".
    debug = False,                                   # Saves all intermediate tables employed during the analysis in pkl format.
    output_path = out_path,                          # Path to save results.
    output_suffix = None                             # Replaces the timestamp in the output files by a user defined string in the  (default: None).
    )

In [ ]:
import pickle
with open("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/cellphoneDB/method1/cpdb_method2_results.pkl", "wb") as f:
    pickle.dump(cpdb_results, f)

In [ ]:
import os
import anndata as ad
import pandas as pd
import ktplotspy as kpy
import matplotlib.pyplot as plt
%matplotlib inline
import pickle

In [ ]:
cpdb_results

In [ ]:
kpy.plot_cpdb_heatmap(pvals = cpdb_results['pvalues'],
                      degs_analysis = False,
                      figsize = (10, 10),
                      title = "Sum of significant interactions")

In [ ]:
adata_scrna

In [ ]:
kpy.plot_cpdb(
    adata = adata_scrna,
    cell_type1 = "mono3",
    cell_type2 = "Fibroblasts",
    means = cpdb_results['means'],
    pvals = cpdb_results['pvalues'],
    celltype_key = "detail_cluster",
    #genes = ["TNF", "IL1B","CCL2","CXCL8"],
    figsize = (10, 3),
    title = "Interactions between mono3 and Fibroblasts",
    max_size = 3,
    highlight_size = 0.75,
    degs_analysis = False,
    standard_scale = True,
    interaction_scores = cpdb_results['interaction_scores'],
    scale_alpha_by_interaction_scores = True
)

In [ ]:
from plotnine import facet_wrap

p = kpy.plot_cpdb(
    adata = adata_scrna,
    cell_type1 = "mono3",
    cell_type2 = "Fibroblasts|T cells|B cells|Endothelial cells",
    means = cpdb_results['means'],
    pvals = cpdb_results['pvalues'],
    celltype_key = "detail_cluster",
    genes = ["TNF", "IL1B","CCL2","CXCL8"],
    figsize = (12, 8),
    title = "Interactions between mono3 and Inflammation \n grouped by classification",
    max_size = 6,
    highlight_size = 0.75,
    degs_analysis = False,
    standard_scale = True,
)
p + facet_wrap("~ classification", ncol = 1)

In [ ]:
from plotnine import facet_wrap

p = kpy.plot_cpdb(
    adata = adata_scrna,
    cell_type1 = "mono1",
    cell_type2 = "Fibroblasts|T cells|B cells|Endothelial cells",
    means = cpdb_results['means'],
    pvals = cpdb_results['pvalues'],
    celltype_key = "detail_cluster",
    genes = ["TNF", "IL1B","CCL2","CXCL8"],
    figsize = (12, 8),
    title = "Interactions between mono1 and Inflammation \n grouped by classification",
    max_size = 6,
    highlight_size = 0.75,
    degs_analysis = False,
    standard_scale = True,
)
p + facet_wrap("~ classification", ncol = 1)

# cellphoneDB method 3 analysis

In [ ]:
import pandas as pd
import sys
import os

In [ ]:
print(sys.version)

In [ ]:
adata=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_cellphoneDB.h5ad")
adata

In [ ]:
adata.obs['state'] = adata.obs['state'].replace({
    'healthy': 'Healthy',
    'NAFLD': 'MASL'
})
adata.obs['detail_cluster'] = adata.obs['detail_cluster'].replace({
    'Fibroblasts': 'Stellete'
})


In [ ]:
adata.obs['detail_cluster'].value_counts()

In [ ]:
adata.obs['state'].value_counts()

In [ ]:
adata.obs['cluster_state'] = adata.obs['detail_cluster'].astype(str) + '-' + adata.obs['state'].astype(str)
adata.obs['cluster_state'].value_counts()

In [ ]:
df = adata.obs[['detail_cluster', 'state']].copy()
df['barcode'] = df.index
df['cluster_state'] = df['detail_cluster'].astype(str) + '-' + df['state'].astype(str)
df_final = df[['barcode', 'cluster_state']]
df_final

In [ ]:
df_final.to_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/cellphoneDB/cellseq_total_cellphneDB.tsv", 
                sep='\t', index=False)

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="cluster_state", method="wilcoxon")

In [ ]:
sc.get.rank_genes_groups_df(adata, group=None).to_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/cellphoneDB/degs_mono3_MASLD_vs_normal.txt", sep="\t", index=False)

In [ ]:
cpdb_file_path = '/home/jaehyunchoi/Tool_testing/cellphoneDB/db/test/v5.0.0/cellphonedb.zip'
meta_file_path = '/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/cellphoneDB/cellseq_total_cellphneDB.tsv'
counts_file_path = '/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_raw.h5ad'
degs_file_path = '/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/cellphoneDB/degs_mono3_MASLD_vs_normal.txt'
out_path = '/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/cellphoneDB/method3'

In [ ]:
cpdb_file_path = '/home/jaehyunchoi/Tool_testing/cellphoneDB/db/test/v5.0.0/cellphonedb.zip'
meta_file_path = '/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/cellphoneDB/cellseq_total_cellphneDB.tsv'
counts_file_path = '/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_raw.h5ad'
degs_file_path = '/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/cellphoneDB/degs_mono3_MASLD_vs_normal.txt'
out_path = '/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/cellphoneDB/method3'

In [ ]:
metadata = pd.read_csv(meta_file_path, sep = '\t')
metadata.head(3)

In [ ]:
pd.read_csv(degs_file_path, sep = '\t') \
    .head(3)

In [ ]:
from cellphonedb.src.core.methods import cpdb_degs_analysis_method

cpdb_results = cpdb_degs_analysis_method.call(
    cpdb_file_path = cpdb_file_path,                            # mandatory: CellphoneDB database zip file.
    meta_file_path = meta_file_path,                            # mandatory: tsv file defining barcodes to cell label.
    counts_file_path = counts_file_path,                        # mandatory: normalized count matrix - a path to the counts file, or an in-memory AnnData object
    degs_file_path = degs_file_path,                            # mandatory: tsv file with DEG to account.
    counts_data = 'hgnc_symbol',                                # defines the gene annotation in counts matrix.
    #active_tfs_file_path = active_tf_path,                      # optional: defines cell types and their active TFs.
    #microenvs_file_path = microenvs_file_path,                  # optional (default: None): defines cells per microenvironment.
    score_interactions = True,                                  # optional: whether to score interactions or not. 
    threshold = 0.1,                                            # defines the min % of cells expressing a gene for this to be employed in the analysis.
    result_precision = 3,                                       # Sets the rounding for the mean values in significan_means.
    separator = '|',                                            # Sets the string to employ to separate cells in the results dataframes "cellA|CellB".
    debug = False,                                              # Saves all intermediate tables emplyed during the analysis in pkl format.
    output_path = out_path,                                     # Path to save results
    output_suffix = None,                                       # Replaces the timestamp in the output files by a user defined string in the  (default: None)
    threads = 25
    )

In [ ]:
from cellphonedb.src.core.methods import cpdb_degs_analysis_method

cpdb_results = cpdb_degs_analysis_method.call(
    cpdb_file_path = cpdb_file_path,                            # mandatory: CellphoneDB database zip file.
    meta_file_path = meta_file_path,                            # mandatory: tsv file defining barcodes to cell label.
    counts_file_path = counts_file_path,                        # mandatory: normalized count matrix - a path to the counts file, or an in-memory AnnData object
    degs_file_path = degs_file_path,                            # mandatory: tsv file with DEG to account.
    counts_data = 'hgnc_symbol',                                # defines the gene annotation in counts matrix.
    #active_tfs_file_path = active_tf_path,                      # optional: defines cell types and their active TFs.
    #microenvs_file_path = microenvs_file_path,                  # optional (default: None): defines cells per microenvironment.
    score_interactions = True,                                  # optional: whether to score interactions or not. 
    threshold = 0.1,                                            # defines the min % of cells expressing a gene for this to be employed in the analysis.
    result_precision = 3,                                       # Sets the rounding for the mean values in significan_means.
    separator = '|',                                            # Sets the string to employ to separate cells in the results dataframes "cellA|CellB".
    debug = False,                                              # Saves all intermediate tables emplyed during the analysis in pkl format.
    output_path = out_path,                                     # Path to save results
    output_suffix = None,                                       # Replaces the timestamp in the output files by a user defined string in the  (default: None)
    threads = 25
    )

In [ ]:
list(cpdb_results.keys())

In [ ]:
list(cpdb_results.keys())

In [ ]:
import matplotlib as mpl

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["pdf.use14corefonts"] = False

In [ ]:
import os
import anndata as ad
import pandas as pd
import ktplotspy as kpy
import matplotlib.pyplot as plt
%matplotlib inline
kpy.plot_cpdb_heatmap(pvals = cpdb_results['relevant_interactions'],
                      degs_analysis=True,
                      figsize=(15, 15),
                      title="Sum of significant interactions")

In [ ]:
import os
import anndata as ad
import pandas as pd
import ktplotspy as kpy
import matplotlib.pyplot as plt
%matplotlib inline
import matplotlib.pyplot as plt

kpy.plot_cpdb_heatmap(
    pvals=cpdb_results['relevant_interactions'],
    degs_analysis=True,
    figsize=(15, 15),
    title="Sum of significant interactions"
)

plt.savefig(
    "/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure/cpdb_heatmap.pdf",
    format="pdf",
    bbox_inches="tight"
)
plt.close()

In [ ]:
adata.obs['cluster_state'].value_counts()

In [ ]:
from plotnine import facet_wrap

p = kpy.plot_cpdb(
    adata = adata,
    cell_type1 = "mono3-Healthy",
    cell_type2 = "T cells-Healthy|B cells-Healthy|Circulating NK/NKT-Healthy|Endothelial cells-Healthy",
    means = cpdb_results['means'],
    pvals = cpdb_results['relevant_interactions'],
    celltype_key = "cluster_state",
    genes = ["TGFB2", "CSF1R", "COL1A1"],
    figsize = (12, 8),
    title = "Interactions between PV and trophoblasts\n grouped by classification",
    max_size = 6,
    highlight_size = 0.75,
    degs_analysis = True,
    standard_scale = True)

p + facet_wrap("~ classification", ncol = 1)

In [ ]:
from plotnine import facet_wrap

p = kpy.plot_cpdb(
    adata = adata,
    cell_type1 = "mono3-MASL",
    cell_type2 = "T cells-MASL|B cells-MASL|Circulating NK/NKT-MASL|Endothelial cells-MASL",
    means = cpdb_results['means'],
    pvals = cpdb_results['relevant_interactions'],
    celltype_key = "cluster_state",
    genes = ["TGFB2", "CSF1R", "COL1A1"],
    figsize = (12, 8),
    title = "Interactions between PV and trophoblasts\n grouped by classification",
    max_size = 6,
    highlight_size = 0.75,
    degs_analysis = True,
    standard_scale = True)

p + facet_wrap("~ classification", ncol = 1)

In [ ]:
df = pd.read_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/cellphoneDB/method3/degs_analysis_significant_means_02_09_2026_173139.txt", sep="\t")

In [ ]:
df

In [ ]:
# 관심 있는 상호작용 쌍
col_healthy = "mono3-Healthy|Stellete-Healthy"
col_disease = "mono3-MASL|Stellete-MASL"

# 둘 다 존재하는 상호작용만 필터링
df_filtered = df[[ "interacting_pair", col_healthy, col_disease ]].dropna()

In [ ]:
df_filtered["diff"] = df_filtered[col_disease] - df_filtered[col_healthy]

# 차이 큰 순으로 정렬
df_filtered_sorted = df_filtered.sort_values(by="diff", ascending=False)

In [ ]:
# 상위 10개 증가 상호작용
print("✅ 질병 상태에서 증가한 상호작용:")
print(df_filtered_sorted.head(10))

# 상위 10개 감소 상호작용
print("⛔ 질병 상태에서 감소한 상호작용:")
print(df_filtered_sorted.tail(10))

In [ ]:
import matplotlib.pyplot as plt

top_pairs = df_filtered_sorted.head(10)["interacting_pair"]
top_df = df_filtered[df_filtered["interacting_pair"].isin(top_pairs)]

top_df.set_index("interacting_pair")[[col_healthy, col_disease]].T.plot(kind="bar", figsize=(10,6))
plt.ylabel("Mean expression")
plt.title("Top 10 Upregulated Interactions in mono3cell ↔ Stelletecell")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
from cellphonedb.utils import search_utils

search_results = search_utils.search_analysis_results(
    query_cell_types_1 = ['mono3-NAFLD'],  # List of cells 1, will be paired to cells 2 (list or 'All').
    query_cell_types_2 = ['T cells-NAFLD', 'B cells-NAFLD', 'Fibroblasts-NAFLD'],     # List of cells 2, will be paired to cells 1 (list or 'All').
    #query_genes = ['TGFBR1'],                                       # filter interactions based on the genes participating (list).
    #query_interactions = ['CSF1_CSF1R'],                            # filter intereactions based on their name (list).
    significant_means = cpdb_results['significant_means'],          # significant_means file generated by CellphoneDB.
    deconvoluted = cpdb_results['deconvoluted'],                    # devonvoluted file generated by CellphoneDB.
    separator = '|',                                                # separator (default: |) employed to split cells (cellA|cellB).
    long_format = True,                                             # converts the output into a wide table, removing non-significant interactions
    query_classifications = ['Signaling by Transforming growth factor']
)

search_results.head(3)

In [ ]:
means_df = pd.read_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/cellphoneDB/method3/degs_analysis_significant_means_01_14_2026_150334.txt", sep="\t")

In [ ]:
means_df['classification'].unique()

In [ ]:
chemokine_df = means_df[means_df['classification'].str.contains("Signaling by Chemokines", case=False, na=False)]
interleukin_df = means_df[means_df['classification'].str.contains("Signaling by Interleukin", case=False, na=False)]

In [ ]:
chemokine_df

In [ ]:
genes = set(chemokine_df["gene_a"].dropna().tolist() + chemokine_df["gene_b"].dropna().tolist())
print(f"추출된 chemokines 관련 유전자 수: {len(genes)}")
interlukin_genes = set(interleukin_df["gene_a"].dropna().tolist() + interleukin_df["gene_b"].dropna().tolist())
print(f"추출된 interleukin_df 관련 유전자 수: {len(genes)}")

In [ ]:
from plotnine import facet_wrap

p = kpy.plot_cpdb(
    adata = adata,
    cell_type1 = "mono3-healthy",
    cell_type2 = "T cells-healthy|B cells-healthy|Circulating NK/NKT-healthy|Endothelial cells-healthy",
    means = cpdb_results['means'],
    pvals = cpdb_results['relevant_interactions'],
    celltype_key = "cluster_state",
    genes = list(genes),  # 위에서 추출한 inflammation 관련 유전자들
    figsize = (12, 8),
    title = "Inflammation-related Interactions between mono3 in MASLD",
    max_size = 6,
    highlight_size = 0.75,
    degs_analysis = True,
    standard_scale = True
)

p + facet_wrap("~ classification", ncol = 1)

In [ ]:
from plotnine import facet_wrap

p = kpy.plot_cpdb(
    adata = adata,
    cell_type1 = "mono3-NAFLD",
    cell_type2 = "T cells-NAFLD|B cells-NAFLD|Circulating NK/NKT-NAFLD|Endothelial cells-NAFLD",
    means = cpdb_results['means'],
    pvals = cpdb_results['relevant_interactions'],
    celltype_key = "cluster_state",
    genes = list(genes),  # 위에서 추출한 inflammation 관련 유전자들
    figsize = (12, 8),
    title = "Inflammation-related Interactions between mono3 in MASLD",
    max_size = 6,
    highlight_size = 0.75,
    degs_analysis = True,
    standard_scale = True
)

p + facet_wrap("~ classification", ncol = 1)

In [ ]:
from plotnine import facet_wrap

p = kpy.plot_cpdb(
    adata = adata,
    cell_type1 = "mono3-healthy",
    cell_type2 = "T cells-healthy|B cells-healthy|Circulating NK/NKT-healthy|Endothelial cells-healthy",
    means = cpdb_results['means'],
    pvals = cpdb_results['relevant_interactions'],
    celltype_key = "cluster_state",
    genes = list(interlukin_genes),
    figsize = (12, 8),
    title = "Inflammation-related Interactions between mono3 in MASLD",
    max_size = 6,
    highlight_size = 0.75,
    degs_analysis = True,
    standard_scale = True
)

p + facet_wrap("~ classification", ncol = 1)

In [ ]:
from plotnine import facet_wrap

p = kpy.plot_cpdb(
    adata = adata,
    cell_type1 = "mono3-NAFLD",
    cell_type2 = "T cells-NAFLD|B cells-NAFLD|Circulating NK/NKT-NAFLD|Endothelial cells-NAFLD",
    means = cpdb_results['means'],
    pvals = cpdb_results['relevant_interactions'],
    celltype_key = "cluster_state",
    genes = list(interlukin_genes),  # 위에서 추출한 inflammation 관련 유전자들
    figsize = (12, 8),
    title = "Inflammation-related Interactions between mono3 in MASLD",
    max_size = 6,
    highlight_size = 0.75,
    degs_analysis = True,
    standard_scale = True
)

p + facet_wrap("~ classification", ncol = 1)

In [ ]:
adata.obs['cluster_state'].value_counts()

In [ ]:
sig_means = cpdb_results['significant_means']
relevant = cpdb_results['relevant_interactions']

cell_healthy = "mono3-healthy|T cells-healthy"
cell_disease = "mono3-NAFLD|T cells-NAFLD"

comp_df = sig_means[["interacting_pair", cell_healthy, cell_disease]].copy()
comp_df = comp_df.dropna(subset=[cell_healthy, cell_disease])

comp_df["delta"] = comp_df[cell_disease] - comp_df[cell_healthy]

In [ ]:
# 질병에서 증가한 상호작용 정렬
up_in_disease = comp_df.sort_values(by="delta", ascending=False)

# 상위 10개 보기
print("질병에서 증가한 상호작용:")
print(up_in_disease.head(10)[["interacting_pair", cell_healthy, cell_disease, "delta"]])

In [ ]:
# 질병에서 감소한 상호작용 정렬
down_in_disease = comp_df.sort_values(by="delta")

print("질병에서 감소한 상호작용:")
print(down_in_disease.head(10)[["interacting_pair", cell_healthy, cell_disease, "delta"]])

# delta NAFLD-healthy Chord diagram

In [ ]:
hv.extension('bokeh')

In [ ]:
import pandas as pd

means = pd.read_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/cellphoneDB/method3/degs_analysis_significant_means_02_09_2026_173139.txt", sep="\t")

In [ ]:
interaction_cols = [col for col in means.columns if "|" in col]
interaction_df = means[["interacting_pair"] + interaction_cols].copy()

In [ ]:
df_long = interaction_df.melt(
    id_vars="interacting_pair",
    value_vars=interaction_cols,
    var_name="pair",
    value_name="weight"
).dropna(subset=["weight"])

df_long[["celltype1", "celltype2"]] = df_long["pair"].str.split("|", expand=True)


In [ ]:
def get_state(cell):
    return "NAFLD" if "NAFLD" in cell else "healthy"

df_long["state1"] = df_long["celltype1"].apply(get_state)
df_long["state2"] = df_long["celltype2"].apply(get_state)

# 순수 healthy / 순수 NAFLD만 남김
df_healthy = df_long.query('state1 == "Healthy" and state2 == "Healthy"')
df_nafld = df_long.query('state1 == "MASL" and state2 == "MASL"')

def clean_name(name):
    return name.replace("-Healthy", "").replace("-MASL", "")

df_healthy["celltype1"] = df_healthy["celltype1"].apply(clean_name)
df_healthy["celltype2"] = df_healthy["celltype2"].apply(clean_name)
df_nafld["celltype1"] = df_nafld["celltype1"].apply(clean_name)
df_nafld["celltype2"] = df_nafld["celltype2"].apply(clean_name)

def make_chord(df, title):
    # unique node list
    nodes = pd.DataFrame({
        "index": pd.unique(df[["celltype1", "celltype2"]].values.ravel("K")),
    })
    nodes["label"] = nodes["index"]

    # edge dataframe
    links = pd.DataFrame({
        "source": df["celltype1"],
        "target": df["celltype2"],
        "value": df["weight"]
    })

    # Convert to hv.Dataset
    node_ds = hv.Dataset(nodes, 'index')
    
    chord = hv.Chord((links, node_ds)).opts(
        opts.Chord(
            title=title,
            labels='label',
            node_color='label',
            edge_color='value',
            cmap='Viridis',
            edge_cmap='Viridis',
            edge_alpha=0.7,
            node_size=15,
            width=650,
            height=650,
            tools=['hover'],
            bgcolor='white'
        )
    )
    return chord


In [ ]:
df_healthy = df_long.query('state1 == "Healthy" and state2 == "Healthy"')
df_nafld = df_long.query('state1 == "MASL" and state2 == "MASL"')

healthy_summary = (
    df_healthy.groupby(["celltype1", "celltype2"])["weight"].mean().reset_index()
)
nafld_summary = (
    df_nafld.groupby(["celltype1", "celltype2"])["weight"].mean().reset_index()
)

In [ ]:
chord_healthy = make_chord(healthy_summary, "Healthy Cell–Cell Interactions")
chord_nafld = make_chord(nafld_summary, "NAFLD Cell–Cell Interactions")

(chord_healthy + chord_nafld).cols(2)

In [ ]:
delta_df = pd.merge(
    nafld_summary,
    healthy_summary,
    on=["celltype1", "celltype2"],
    suffixes=("_nafld", "_healthy"),
    how="outer"
).fillna(0)

delta_df["delta_weight"] = delta_df["weight_nafld"] - delta_df["weight_healthy"]

In [ ]:
delta_df

In [ ]:
delta_df = delta_df[delta_df["celltype1"] != delta_df["celltype2"]]
delta_df["pair_sorted"] = delta_df.apply(
    lambda x: "-".join(sorted([x["celltype1"], x["celltype2"]])), axis=1
)
delta_df = (
    delta_df.groupby("pair_sorted")["delta_weight"]
    .mean()
    .reset_index()
    .assign(celltype1=lambda x: x["pair_sorted"].str.split("-").str[0],
            celltype2=lambda x: x["pair_sorted"].str.split("-").str[2])
)
delta_df["abs_value"] = delta_df["delta_weight"].abs()
top_n = 100
delta_df = delta_df.reindex(delta_df["delta_weight"].abs().sort_values(ascending=False).index)
#delta_df = delta_df.head(top_n)

In [ ]:
delta_df

In [ ]:
nodes = pd.DataFrame({
    "index": pd.unique(delta_df[["celltype1", "celltype2"]].values.ravel("K"))
})
nodes["label"] = nodes["index"]

links = pd.DataFrame({
    "source": delta_df["celltype1"],
    "target": delta_df["celltype2"],
    "value": delta_df["abs_value"],        # layout stability
    "color_value": delta_df["delta_weight"] # color encodes direction
})

node_ds = hv.Dataset(nodes, "index")

In [ ]:
links

In [ ]:
delta_chord = hv.Chord((links, node_ds)).opts(
    opts.Chord(
        title="Δ(weight) — NAFLD minus Healthy",
        labels="label",
        node_color="label",
        edge_color="color_value",   # color by signed Δ(weight)
        cmap="rainbow",              # red = gain, blue = loss
        edge_cmap="RdBu_r",
        colorbar=True,
        edge_alpha=0.8,
        node_size=15,
        width=750,
        height=750,
        tools=["hover"],
        bgcolor="white"
    )
)

delta_chord

# Mono3 - Stellete interaction

In [ ]:
# Mono3 Source interaction

In [ ]:
means = pd.read_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/cellphoneDB/method3/degs_analysis_significant_means_02_09_2026_173139.txt", sep="\t")

In [ ]:
means

In [ ]:
interaction_cols = [col for col in means.columns if "|" in col]
interaction_df = means[["interacting_pair"] + interaction_cols].copy()

In [ ]:
interaction_df

In [ ]:
def is_mono3(col):
    a, b = col.split("|")
    return ("mono3" in a )

tb_cols = [c for c in interaction_cols if is_mono3(c)]
tb_df = means[["interacting_pair"] + tb_cols].copy()
tb_df

In [ ]:
df_long = tb_df.melt(
    id_vars="interacting_pair",
    var_name="pair",
    value_name="weight"
).dropna(subset=["weight"])

df_long[["celltype1", "celltype2"]] = df_long["pair"].str.split("|", expand=True)

# 상태 정보
def get_state(cell):
    return "MASL" if "MASL" in cell else "Healthy"

df_long["state1"] = df_long["celltype1"].apply(get_state)
df_long["state2"] = df_long["celltype2"].apply(get_state)

# 이름 정리
def clean_name(name):
    return name.replace("-Healthy", "").replace("-MASL", "")

for c in ["celltype1", "celltype2"]:
    df_long[c] = df_long[c].apply(clean_name)

In [ ]:
df_long

In [ ]:
df_healthy = df_long.query('state1 == "Healthy" and state2 == "Healthy"')
df_nafld = df_long.query('state1 == "MASL" and state2 == "MASL"')

healthy_summary = (
    df_healthy.groupby(["interacting_pair", "celltype2"])["weight"].mean().reset_index()
)
nafld_summary = (
    df_nafld.groupby(["interacting_pair", "celltype2"])["weight"].mean().reset_index()
)

In [ ]:
delta = pd.merge(
    nafld_summary, healthy_summary,
    on=["interacting_pair", "celltype2"],
    suffixes=("_MASL", "_Healthy"),
    how="outer"
).fillna(0)

delta["delta_weight"] = delta["weight_MASL"] - delta["weight_Healthy"]

In [ ]:
delta

In [ ]:
pivot_df = delta.pivot_table(
    index="interacting_pair",
    columns="celltype2",
    values="delta_weight",
    fill_value=0
)

In [ ]:
pivot_df.to_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure/Mono3_source_cellphoneDB.csv")

In [ ]:
plt.figure(figsize=(10, 40))
sns.heatmap(
    pivot_df,
    cmap="RdBu_r",
    center=0,
    linewidths=0.5,
    cbar_kws={'label': 'Δ(weight) (MASL - Healthy)'}
)
plt.title("Mono3 → Target Cell Interactions (Δweight: MASL - Healthy)", fontsize=14)
plt.xlabel("Target cell type")
plt.ylabel("Ligand–Receptor Pair (Mono3 source)")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 40))
sns.heatmap(
    pivot_df,
    cmap="RdBu_r",
    center=0,
    linewidths=0.5,
    cbar_kws={'label': 'Δ(weight) (MASL - Healthy)'}
)
plt.title("Mono3 → Target Cell Interactions (Δweight: MASL - Healthy)", fontsize=14)
plt.xlabel("Target cell type")
plt.ylabel("Ligand–Receptor Pair (Mono3 source)")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 40))
sns.heatmap(
    pivot_df,
    cmap="RdBu_r",
    center=0,
    linewidths=0.5,
    cbar_kws={'label': 'Δ(weight) (MASL - Healthy)'}
)

plt.title(
    "Mono3 → Target Cell Interactions (Δweight: MASL - Healthy)",
    fontsize=14
)
plt.xlabel("Target cell type")
plt.ylabel("Ligand–Receptor Pair (Mono3 source)")

plt.tight_layout()

# PDF 저장
plt.savefig(
    "/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure/Mono3_interactions_heatmap.pdf",
    format="pdf",
    bbox_inches="tight"
)

plt.show()
plt.close()

In [ ]:
upper_thr = delta["delta_weight"].quantile(0.9)
lower_thr = delta["delta_weight"].quantile(0.1)

In [ ]:
top10 = delta.query("delta_weight >= @upper_thr")
bottom10 = delta.query("delta_weight <= @lower_thr")

In [ ]:
top10

In [ ]:
pivot_top10 = top10.pivot_table(
    index="interacting_pair",
    columns="celltype2",
    values="delta_weight",
    fill_value=0
)

In [ ]:
pivot_bottom10 = bottom10.pivot_table(
    index="interacting_pair",
    columns="celltype2",
    values="delta_weight",
    fill_value=0
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 0.4 * pivot_top10.shape[0]))

sns.heatmap(
    pivot_top10,
    cmap="Reds",
    linewidths=0.5,
    cbar_kws={"label": "Δ(weight) (MASL - Healthy)"}
)

plt.title("Top 10% MASL-enriched interactions", fontsize=14)
plt.xlabel("Target cell type")
plt.ylabel("Ligand–Receptor Pair")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 0.4 * pivot_bottom10.shape[0]))

sns.heatmap(
    pivot_bottom10,
    cmap="Blues_r",
    linewidths=0.5,
    cbar_kws={"label": "Δ(weight) (MASL - Healthy)"}
)

plt.title("Bottom 10% Healthy-enriched interactions", fontsize=14)
plt.xlabel("Target cell type")
plt.ylabel("Ligand–Receptor Pair")
plt.tight_layout()
plt.show()

In [ ]:
# mono3 target analysis

In [ ]:
means = pd.read_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/cellphoneDB/method3/degs_analysis_significant_means_02_09_2026_173139.txt", sep="\t")
interaction_cols = [col for col in means.columns if "|" in col]
interaction_df = means[["interacting_pair"] + interaction_cols].copy()

In [ ]:
def is_mono3(col):
    a, b = col.split("|")
    return ("mono3" in b )

tb_cols = [c for c in interaction_cols if is_mono3(c)]
tb_df = means[["interacting_pair"] + tb_cols].copy()
tb_df

In [ ]:
df_long = tb_df.melt(
    id_vars="interacting_pair",
    var_name="pair",
    value_name="weight"
).dropna(subset=["weight"])

df_long[["celltype1", "celltype2"]] = df_long["pair"].str.split("|", expand=True)
# 상태 정보
def get_state(cell):
    return "MASL" if "MASL" in cell else "Healthy"

df_long["state1"] = df_long["celltype1"].apply(get_state)
df_long["state2"] = df_long["celltype2"].apply(get_state)

# 이름 정리
def clean_name(name):
    return name.replace("-Healthy", "").replace("-MASL", "")

for c in ["celltype1", "celltype2"]:
    df_long[c] = df_long[c].apply(clean_name)

df_long

In [ ]:
df_healthy = df_long.query('state1 == "Healthy" and state2 == "Healthy"')
df_nafld = df_long.query('state1 == "MASL" and state2 == "MASL"')

healthy_summary = (
    df_healthy.groupby(["interacting_pair", "celltype1"])["weight"].mean().reset_index()
)
nafld_summary = (
    df_nafld.groupby(["interacting_pair", "celltype1"])["weight"].mean().reset_index()
)

In [ ]:
delta = pd.merge(
    nafld_summary, healthy_summary,
    on=["interacting_pair", "celltype1"],
    suffixes=("_MASL", "_Healthy"),
    how="outer"
).fillna(0)

delta["delta_weight"] = delta["weight_MASL"] - delta["weight_Healthy"]
delta

In [ ]:
pivot_df = delta.pivot_table(
    index="interacting_pair",
    columns="celltype1",
    values="delta_weight",
    fill_value=0
)

In [ ]:
pivot_df

In [ ]:
pivot_df.to_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure/Mono3_target_cellphoneDB.csv")

In [ ]:
plt.figure(figsize=(10, 40))
sns.heatmap(
    pivot_df,
    cmap="RdBu_r",
    center=0,
    linewidths=0.5,
    cbar_kws={'label': 'Δ(weight) (MASL - Healthy)'}
)
plt.title("Source cell → mono3 Interactions (Δweight: MASL - Healthy)", fontsize=14)
plt.xlabel("Target cell type")
plt.ylabel("Ligand–Receptor Pair (Mono3 source)")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 40))
sns.heatmap(
    pivot_df,
    cmap="RdBu_r",
    center=0,
    linewidths=0.5,
    cbar_kws={'label': 'Δ(weight) (MASL - Healthy)'}
)

plt.title(
    "Source Cell → Mono3 Cell Interactions (Δweight: MASL - Healthy)",
    fontsize=14
)
plt.xlabel("Target cell type")
plt.ylabel("Ligand–Receptor Pair (Mono3 target)")

plt.tight_layout()

# PDF 저장
plt.savefig(
    "/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure/Mono3_target_interactions_heatmap.pdf",
    format="pdf",
    bbox_inches="tight"
)

plt.show()
plt.close()

In [ ]:
delta["delta_weight"].describe()

In [ ]:
upper_thr = delta["delta_weight"].quantile(0.9)
lower_thr = delta["delta_weight"].quantile(0.1)

upper_thr, lower_thr

In [ ]:
top10 = delta.query("delta_weight >= @upper_thr")
bottom10 = delta.query("delta_weight <= @lower_thr")

In [ ]:
pivot_top10 = top10.pivot_table(
    index="interacting_pair",
    columns="celltype1",
    values="delta_weight",
    fill_value=0
)

In [ ]:
pivot_bottom10 = bottom10.pivot_table(
    index="interacting_pair",
    columns="celltype1",
    values="delta_weight",
    fill_value=0
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 0.4 * pivot_top10.shape[0]))

sns.heatmap(
    pivot_top10,
    cmap="Reds",
    linewidths=0.5,
    cbar_kws={"label": "Δ(weight) (MASL - Healthy)"}
)

plt.title("Top 10% MASL-enriched interactions", fontsize=14)
plt.xlabel("Target cell type")
plt.ylabel("Ligand–Receptor Pair")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 0.4 * pivot_bottom10.shape[0]))

sns.heatmap(
    pivot_bottom10,
    cmap="Blues_r",
    linewidths=0.5,
    cbar_kws={"label": "Δ(weight) (MASL - Healthy)"}
)

plt.title("Bottom 10% Healthy-enriched interactions", fontsize=14)
plt.xlabel("Target cell type")
plt.ylabel("Ligand–Receptor Pair")
plt.tight_layout()
plt.show()

# Mono3 - T cells interaction

In [ ]:
means = pd.read_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/cellphoneDB/method3/degs_analysis_significant_means_02_09_2026_173139.txt", sep="\t")

In [ ]:
interaction_cols = [col for col in means.columns if "|" in col]
interaction_df = means[["interacting_pair"] + interaction_cols].copy()

In [ ]:
def is_mono3(col):
    a, b = col.split("|")
    return ("mono3" in a )

tbout_cols = [c for c in interaction_cols if is_mono3(c)]
tbout_df = means[["interacting_pair"] + tbout_cols].copy()
tbout_df

In [ ]:
dfout_long = tbout_df.melt(
    id_vars="interacting_pair",
    var_name="pair",
    value_name="weight"
).dropna(subset=["weight"])

dfout_long[["celltype1", "celltype2"]] = dfout_long["pair"].str.split("|", expand=True)

# 상태 정보
def get_state(cell):
    return "MASL" if "MASL" in cell else "Healthy"

dfout_long["state1"] = dfout_long["celltype1"].apply(get_state)
dfout_long["state2"] = dfout_long["celltype2"].apply(get_state)

# 이름 정리
def clean_name(name):
    return name.replace("-Healthy", "").replace("-MASL", "")

for c in ["celltype1", "celltype2"]:
    dfout_long[c] = dfout_long[c].apply(clean_name)

In [ ]:
dfout_healthy = dfout_long.query('state1 == "Healthy" and state2 == "Healthy"')
dfout_nafld = dfout_long.query('state1 == "MASL" and state2 == "MASL"')

healthyout_summary = (
    dfout_healthy.groupby(["interacting_pair", "celltype2"])["weight"].mean().reset_index()
)
nafldout_summary = (
    dfout_nafld.groupby(["interacting_pair", "celltype2"])["weight"].mean().reset_index()
)

delta_out = pd.merge(
    nafldout_summary, healthyout_summary,
    on=["interacting_pair", "celltype2"],
    suffixes=("_MASL", "_Healthy"),
    how="outer"
).fillna(0)

delta_out["delta_weight"] = delta_out["weight_MASL"] - delta_out["weight_Healthy"]

In [ ]:
pivot_out_df = delta_out.pivot_table(
    index="interacting_pair",
    columns="celltype2",
    values="delta_weight",
    fill_value=0
)
pivot_out_df.to_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure/Mono3_source_cellphoneDB.csv")

# Mono4 interaction

In [ ]:
means = pd.read_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/cellphoneDB/method3/degs_analysis_significant_means_02_09_2026_173139.txt", sep="\t")
interaction_cols = [col for col in means.columns if "|" in col]
interaction_df = means[["interacting_pair"] + interaction_cols].copy()

In [ ]:
def is_KC2(col):
    a, b = col.split("|")
    return ("KC2" in a )

tbout_cols = [c for c in interaction_cols if is_KC2(c)]
tbout_df = means[["interacting_pair"] + tbout_cols].copy()
tbout_df

In [ ]:
dfout_long = tbout_df.melt(
    id_vars="interacting_pair",
    var_name="pair",
    value_name="weight"
).dropna(subset=["weight"])

dfout_long[["celltype1", "celltype2"]] = dfout_long["pair"].str.split("|", expand=True)

# 상태 정보
def get_state(cell):
    return "MASL" if "MASL" in cell else "Healthy"

dfout_long["state1"] = dfout_long["celltype1"].apply(get_state)
dfout_long["state2"] = dfout_long["celltype2"].apply(get_state)

# 이름 정리
def clean_name(name):
    return name.replace("-Healthy", "").replace("-MASL", "")

for c in ["celltype1", "celltype2"]:
    dfout_long[c] = dfout_long[c].apply(clean_name)

In [ ]:
dfout_healthy = dfout_long.query('state1 == "Healthy" and state2 == "Healthy"')
dfout_nafld = dfout_long.query('state1 == "MASL" and state2 == "MASL"')

healthyout_summary = (
    dfout_healthy.groupby(["interacting_pair", "celltype2"])["weight"].mean().reset_index()
)
nafldout_summary = (
    dfout_nafld.groupby(["interacting_pair", "celltype2"])["weight"].mean().reset_index()
)

delta_out = pd.merge(
    nafldout_summary, healthyout_summary,
    on=["interacting_pair", "celltype2"],
    suffixes=("_MASL", "_Healthy"),
    how="outer"
).fillna(0)

delta_out["delta_weight"] = delta_out["weight_MASL"] - delta_out["weight_Healthy"]

In [ ]:
delta_out

In [ ]:
pivot_df = delta_out.pivot_table(
    index="interacting_pair",
    columns="celltype2",
    values="delta_weight",
    fill_value=0
)

In [ ]:
pivot_df.to_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure/KC2_Source_cellphoneDB.csv")

In [ ]:
plt.figure(figsize=(10, 40))
sns.heatmap(
    pivot_df,
    cmap="RdBu_r",
    center=0,
    linewidths=0.5,
    cbar_kws={'label': 'Δ(weight) (MASL - Healthy)'}
)
plt.title("KC2 → Target cells Interactions (Δweight: MASL - Healthy)", fontsize=14)
plt.xlabel("Target cell type")
plt.ylabel("Ligand–Receptor Pair (KC2 source)")
plt.tight_layout()
plt.show()

In [ ]:
means = pd.read_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/cellphoneDB/method3/degs_analysis_significant_means_02_09_2026_173139.txt", sep="\t")
interaction_cols = [col for col in means.columns if "|" in col]
interaction_df = means[["interacting_pair"] + interaction_cols].copy()

def is_KC2(col):
    a, b = col.split("|")
    return ("KC2" in b )

tbout_cols = [c for c in interaction_cols if is_KC2(c)]
tbout_df = means[["interacting_pair"] + tbout_cols].copy()
tbout_df

In [ ]:
dfout_long = tbout_df.melt(
    id_vars="interacting_pair",
    var_name="pair",
    value_name="weight"
).dropna(subset=["weight"])

dfout_long[["celltype1", "celltype2"]] = dfout_long["pair"].str.split("|", expand=True)

# 상태 정보
def get_state(cell):
    return "MASL" if "MASL" in cell else "Healthy"

dfout_long["state1"] = dfout_long["celltype1"].apply(get_state)
dfout_long["state2"] = dfout_long["celltype2"].apply(get_state)

# 이름 정리
def clean_name(name):
    return name.replace("-Healthy", "").replace("-MASL", "")

for c in ["celltype1", "celltype2"]:
    dfout_long[c] = dfout_long[c].apply(clean_name)

In [ ]:
dfin_healthy = dfout_long.query('state1 == "Healthy" and state2 == "Healthy"')
dfin_nafld = dfout_long.query('state1 == "MASL" and state2 == "MASL"')

healthyin_summary = (
    dfin_healthy.groupby(["interacting_pair", "celltype1"])["weight"].mean().reset_index()
)
nafldin_summary = (
    dfin_nafld.groupby(["interacting_pair", "celltype1"])["weight"].mean().reset_index()
)

delta_in = pd.merge(
    nafldin_summary, healthyin_summary,
    on=["interacting_pair", "celltype1"],
    suffixes=("_MASL", "_Healthy"),
    how="outer"
).fillna(0)

delta_in["delta_weight"] = delta_in["weight_MASL"] - delta_in["weight_Healthy"]

In [ ]:
delta_in

In [ ]:
pivot_df = delta_in.pivot_table(
    index="interacting_pair",
    columns="celltype1",
    values="delta_weight",
    fill_value=0
)

In [ ]:
pivot_df.to_csv("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure/KC2_Target_cellphoneDB.csv")

In [ ]:
plt.figure(figsize=(10, 40))
sns.heatmap(
    pivot_df,
    cmap="RdBu_r",
    center=0,
    linewidths=0.5,
    cbar_kws={'label': 'Δ(weight) (MASL - Healthy)'}
)

plt.title(
    "Source Cell → KC2 Cell Interactions (Δweight: MASL - Healthy)",
    fontsize=14
)
plt.xlabel("Target cell type")
plt.ylabel("Ligand–Receptor Pair (KC2 target)")

plt.tight_layout()

# PDF 저장
#plt.savefig(
#    "/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure/Mono3_target_interactions_heatmap.pdf",
#    format="pdf",
#    bbox_inches="tight"
#)

plt.show()
plt.close()

In [ ]:
upper_thr = delta_out["delta_weight"].quantile(0.9)
lower_thr = delta_out["delta_weight"].quantile(0.1)

upper_thr, lower_thr

In [ ]:
upper_thr = delta_in["delta_weight"].quantile(0.9)
lower_thr = delta_in["delta_weight"].quantile(0.1)

upper_thr, lower_thr

In [ ]:
top10 = delta_in.query("delta_weight >= @upper_thr")
bottom10 = delta_in.query("delta_weight <= @lower_thr")

In [ ]:
pivot_top10 = top10.pivot_table(
    index="interacting_pair",
    columns="celltype1",
    values="delta_weight",
    fill_value=0
)

In [ ]:
pivot_bottom10 = bottom10.pivot_table(
    index="interacting_pair",
    columns="celltype1",
    values="delta_weight",
    fill_value=0
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 0.4 * pivot_top10.shape[0]))

sns.heatmap(
    pivot_top10,
    cmap="Reds",
    linewidths=0.5,
    cbar_kws={"label": "Δ(weight) (MASL - Healthy)"}
)

plt.title("Top 10% MASL-enriched interactions", fontsize=14)
plt.xlabel("Source cell type")
plt.ylabel("Ligand–Receptor Pair")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 0.4 * pivot_bottom10.shape[0]))

sns.heatmap(
    pivot_bottom10,
    cmap="Blues_r",
    linewidths=0.5,
    cbar_kws={"label": "Δ(weight) (MASL - Healthy)"}
)

plt.title("Bottom 10% Healthy-enriched interactions", fontsize=14)
plt.xlabel("Source cell type")
plt.ylabel("Ligand–Receptor Pair")
plt.tight_layout()
plt.show()